# Beautiful Tissue Gallery

This gallery showcases the full capabilities of PointillSim through stunning visualizations of simulated tissues. Each example demonstrates realistic tissue architecture with publication-quality graphics.

---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse, Polygon, Circle, FancyBboxPatch
from matplotlib.collections import PatchCollection
import matplotlib.colors as mcolors
from matplotlib.colors import LinearSegmentedColormap
from scipy.ndimage import gaussian_filter

np.random.seed(42)

from pointillsim import (
    TissueCellTypes,
    CellTypesProperties,
    HybISS_Setup,
    FOVDistribution,
    FrameWideElement,
    VacuolatedStructure,
    LinearLumenStructure,
    RandomCellTypeRule,
    MixOfNCellTypesRule,
    SingleTypeRule,
    DistanceBasedRule,
    ProbabilityNodeFieldRule,
)

from pointillsim.rules.composite import LayerRule, GradientRule, CompositeRule
from pointillsim.elements import (
    LayeredElement,
    BranchingStructure,
    FibrillarStructure,
    ClusterElement,
    GlandularUnit,
    InterfaceElement,
    StromalElement,
)

# Beautiful style settings
plt.rcParams['figure.dpi'] = 150
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = '#f8f8f8'
plt.rcParams['font.family'] = 'sans-serif'

print("Gallery ready!")

In [ ]:
# Beautiful visualization function
def render_tissue(fov, cell_props, title, 
                  figsize=(12, 12), 
                  show_nuclei=True,
                  cell_alpha=0.7,
                  edge_width=0.3,
                  background_color='#fafafa',
                  legend_names=None,
                  colormap=None):
    """
    Create a publication-quality tissue visualization.
    """
    fig, ax = plt.subplots(figsize=figsize)
    ax.set_facecolor(background_color)
    
    # Apply cell properties if not already applied
    if not hasattr(fov, 'cell_colors') or fov.cell_colors is None:
        cell_props.apply(fov)
    
    # Custom colors if provided
    if colormap is not None:
        n_types = len(np.unique(fov.class_instance))
        custom_colors = colormap(np.linspace(0, 1, n_types))
        cell_colors = [custom_colors[int(c)] for c in fov.class_instance]
    else:
        cell_colors = fov.cell_colors
    
    # Draw cells as ellipses
    ellipses = []
    colors = []
    
    for i in range(len(fov.cell_centroids)):
        ellipse = Ellipse(
            xy=(fov.cell_centroids[i, 0], fov.cell_centroids[i, 1]),
            width=2 * fov.cell_major_axis[i],
            height=2 * fov.cell_minor_axis[i],
            angle=np.degrees(fov.cell_rotation[i]),
        )
        ellipses.append(ellipse)
        colors.append(cell_colors[i])
    
    collection = PatchCollection(ellipses, alpha=cell_alpha)
    collection.set_facecolors(colors)
    collection.set_edgecolors('black')
    collection.set_linewidths(edge_width)
    ax.add_collection(collection)
    
    # Draw nuclei
    if show_nuclei:
        for i in range(len(fov.cell_centroids)):
            nucleus = Circle(
                (fov.cell_centroids[i, 0], fov.cell_centroids[i, 1]),
                radius=fov.cell_minor_axis[i] * 0.4,
                facecolor='#333333',
                edgecolor='none',
                alpha=0.4,
            )
            ax.add_patch(nucleus)
    
    # Frame settings
    frame_size = fov.frame_size if hasattr(fov, 'frame_size') else max(fov.cell_centroids.max(axis=0))
    ax.set_xlim(0, frame_size)
    ax.set_ylim(0, frame_size)
    ax.set_aspect('equal')
    ax.set_xticks([])
    ax.set_yticks([])
    
    # Title
    ax.set_title(title, fontsize=16, fontweight='bold', pad=20)
    
    # Legend
    if legend_names is not None:
        from matplotlib.patches import Patch
        unique_types = np.unique(fov.class_instance)
        legend_elements = []
        for i, t in enumerate(unique_types):
            if colormap is not None:
                color = custom_colors[i]
            else:
                color = cell_props.colordict.get(int(t), 'gray')
            name = legend_names[int(t)] if int(t) < len(legend_names) else f'Type {t}'
            legend_elements.append(Patch(facecolor=color, alpha=cell_alpha, label=name))
        ax.legend(handles=legend_elements, loc='upper right', fontsize=10, 
                  framealpha=0.9, edgecolor='gray')
    
    # Cell count annotation
    ax.text(0.02, 0.02, f'{len(fov.cell_centroids)} cells', 
            transform=ax.transAxes, fontsize=10, 
            color='gray', style='italic')
    
    plt.tight_layout()
    return fig, ax

---
## 1. Colon Tissue with Crypts

Beautiful colonic epithelium with multiple crypt structures showing stem-to-differentiated gradients.

In [ ]:
# Colon tissue parameters
n_cell_types_colon = 6
frame_size_colon = 1000

tissue_colon = TissueCellTypes()
tissue_colon.generate_types_and_markers(n_genes=50, n_cell_types=n_cell_types_colon)
cell_type_names_colon = ['Stem Cell', 'Transit-Amplifying', 'Enterocyte', 'Goblet', 'Fibroblast', 'Immune']

# Beautiful custom colors for colon
colon_colors = {
    0: '#FF6B6B',  # Stem - Red
    1: '#FFA07A',  # Transit - Salmon
    2: '#98D8C8',  # Enterocyte - Teal
    3: '#7B68EE',  # Goblet - Purple
    4: '#F7DC6F',  # Fibroblast - Yellow
    5: '#85C1E9',  # Immune - Blue
}

cell_props_colon = CellTypesProperties(
    n_cell_types=n_cell_types_colon,
    sizes=[8, 10, 12, 14, 14, 8],
    anisotropy=[0.9, 0.85, 0.8, 0.75, 0.7, 0.95],
    colordict=colon_colors,
)

# Crypt structure
def colon_crypt():
    rule = LayerRule(
        n_cell_types=n_cell_types_colon,
        layer_types=[0, 1, 2],  # Stem → Transit → Enterocyte
        layer_boundaries=[0.3, 0.65],
        transition_width=10,
    )
    return VacuolatedStructure(
        frame_size=frame_size_colon,
        scale=50 + np.random.uniform(-10, 15),
        hole_scale_factor=0.5,
        rules=rule,
        tipical_cell_spacing=10,
    )

# Background stroma
def colon_stroma():
    return FrameWideElement(
        frame_size=frame_size_colon,
        tipical_cell_spacing=25,
        rules=MixOfNCellTypesRule(
            n_cell_types=n_cell_types_colon,
            list_N=[4, 5],
            proportions=[0.7, 0.3]
        )
    )

# Generate
np.random.seed(42)
fov_dist_colon = FOVDistribution(
    frame_size=frame_size_colon,
    background_element=colon_stroma,
    other_elements=[colon_crypt],
    elements_frequency=[1.0],
    attempts_at_elements=20,
)

fov_colon = fov_dist_colon.generate_fov()

# Render
fig, ax = render_tissue(
    fov_colon, cell_props_colon,
    title='Colonic Epithelium with Crypts',
    legend_names=cell_type_names_colon,
    figsize=(14, 14),
    cell_alpha=0.75,
)
plt.show()

---
## 2. Cerebral Cortex with Layers

Stratified cortical tissue showing distinct neuronal layers.

In [ ]:
# Cortex parameters
n_cell_types_cortex = 7
frame_size_cortex = 1000

tissue_cortex = TissueCellTypes()
tissue_cortex.generate_types_and_markers(n_genes=60, n_cell_types=n_cell_types_cortex)
cell_type_names_cortex = ['L1', 'L2/3 Pyramidal', 'L4 Granular', 'L5 Pyramidal', 'L6 Pyramidal', 'Interneuron', 'Astrocyte']

# Gradient colors for layers
cortex_colors = {
    0: '#E8DAEF',  # L1 - Light purple
    1: '#BB8FCE',  # L2/3 - Medium purple
    2: '#76448A',  # L4 - Dark purple
    3: '#1A5276',  # L5 - Dark blue
    4: '#2874A6',  # L6 - Medium blue
    5: '#F39C12',  # Interneuron - Orange
    6: '#27AE60',  # Astrocyte - Green
}

cell_props_cortex = CellTypesProperties(
    n_cell_types=n_cell_types_cortex,
    sizes=[8, 14, 10, 18, 14, 10, 12],
    anisotropy=[0.9, 0.6, 0.85, 0.5, 0.65, 0.9, 0.75],
    colordict=cortex_colors,
)

# Layer-specific probability field
reference_points = np.array([
    [200, 50], [500, 50], [800, 50],      # L1
    [200, 180], [500, 180], [800, 180],   # L2/3
    [200, 380], [500, 380], [800, 380],   # L4
    [200, 580], [500, 580], [800, 580],   # L5
    [200, 780], [500, 780], [800, 780],   # L6
    [200, 950], [500, 950], [800, 950],   # White matter
])

def softmax(x):
    exp_x = np.exp(x - np.max(x, axis=1, keepdims=True))
    return exp_x / np.sum(exp_x, axis=1, keepdims=True)

# Layer composition (logits)
logits = np.array([
    # L1: mostly astrocytes, some interneurons
    [2, 0, 0, 0, 0, 1, 3], [2, 0, 0, 0, 0, 1, 3], [2, 0, 0, 0, 0, 1, 3],
    # L2/3: pyramidal neurons
    [0, 4, 0, 0, 0, 1.5, 0.5], [0, 4, 0, 0, 0, 1.5, 0.5], [0, 4, 0, 0, 0, 1.5, 0.5],
    # L4: granular neurons
    [0, 0.5, 4, 0.5, 0, 1, 0.5], [0, 0.5, 4, 0.5, 0, 1, 0.5], [0, 0.5, 4, 0.5, 0, 1, 0.5],
    # L5: large pyramidal
    [0, 0.5, 0.5, 4, 0.5, 1, 0.5], [0, 0.5, 0.5, 4, 0.5, 1, 0.5], [0, 0.5, 0.5, 4, 0.5, 1, 0.5],
    # L6: pyramidal
    [0, 0, 0, 0.5, 4, 1, 0.5], [0, 0, 0, 0.5, 4, 1, 0.5], [0, 0, 0, 0.5, 4, 1, 0.5],
    # White matter: mostly astrocytes
    [0, 0, 0, 0, 0, 0.5, 4], [0, 0, 0, 0, 0, 0.5, 4], [0, 0, 0, 0, 0, 0.5, 4],
])

cortex_rule = ProbabilityNodeFieldRule(
    n_cell_types=n_cell_types_cortex,
    n_ref_points=len(reference_points),
    ref_probs=softmax(logits),
    reference_points=reference_points,
)

# Generate cortex
np.random.seed(42)
fov_dist_cortex = FOVDistribution(
    frame_size=frame_size_cortex,
    background_element=lambda: FrameWideElement(
        frame_size=frame_size_cortex,
        tipical_cell_spacing=18,
        rules=cortex_rule,
    ),
)

fov_cortex = fov_dist_cortex.generate_fov()

# Render
fig, ax = render_tissue(
    fov_cortex, cell_props_cortex,
    title='Cerebral Cortex: Layered Structure',
    legend_names=cell_type_names_cortex,
    figsize=(12, 14),
    cell_alpha=0.8,
    show_nuclei=True,
)

# Add layer annotations
layer_positions = [50, 180, 380, 580, 780, 950]
layer_names = ['L1', 'L2/3', 'L4', 'L5', 'L6', 'WM']
for y, name in zip(layer_positions, layer_names):
    ax.text(1020, y, name, fontsize=12, va='center', fontweight='bold', color='#333')
    ax.axhline(y - 50, xmin=0.95, xmax=1.0, color='gray', linewidth=0.5, alpha=0.5)

plt.show()

---
## 3. Mammary Gland Tissue

Glandular tissue with acini, ducts, and surrounding stroma.

In [ ]:
# Mammary gland parameters
n_cell_types_breast = 6
frame_size_breast = 1000

tissue_breast = TissueCellTypes()
tissue_breast.generate_types_and_markers(n_genes=45, n_cell_types=n_cell_types_breast)
cell_type_names_breast = ['Luminal', 'Myoepithelial', 'Fibroblast', 'Adipocyte', 'Endothelial', 'Macrophage']

# Warm colors for breast tissue
breast_colors = {
    0: '#E74C3C',  # Luminal - Red
    1: '#E67E22',  # Myoepithelial - Orange
    2: '#F4D03F',  # Fibroblast - Yellow
    3: '#FAD7A0',  # Adipocyte - Light cream
    4: '#3498DB',  # Endothelial - Blue
    5: '#9B59B6',  # Macrophage - Purple
}

cell_props_breast = CellTypesProperties(
    n_cell_types=n_cell_types_breast,
    sizes=[12, 10, 14, 30, 10, 10],
    anisotropy=[0.85, 0.65, 0.75, 0.95, 0.9, 0.9],
    colordict=breast_colors,
)

# Acinus with bilayer
def breast_acinus():
    rule = LayerRule(
        n_cell_types=n_cell_types_breast,
        layer_types=[0, 1],  # Luminal inner, Myoepithelial outer
        layer_boundaries=[0.6],
        transition_width=8,
    )
    return VacuolatedStructure(
        frame_size=frame_size_breast,
        scale=60 + np.random.uniform(-15, 20),
        hole_scale_factor=0.55,
        rules=rule,
        tipical_cell_spacing=10,
    )

# Blood vessel
def breast_vessel():
    return LinearLumenStructure(
        frame_size=frame_size_breast,
        start_point=(np.random.uniform(50, 950), np.random.uniform(50, 950)),
        end_point=(np.random.uniform(50, 950), np.random.uniform(50, 950)),
        width=25,
        lumen_fraction=0.6,
        tipical_cell_spacing=8,
        rules=SingleTypeRule(n_cell_types=n_cell_types_breast, cell_type_ix=4),
    )

# Stromal background with adipocytes
def breast_stroma():
    return FrameWideElement(
        frame_size=frame_size_breast,
        tipical_cell_spacing=30,
        rules=MixOfNCellTypesRule(
            n_cell_types=n_cell_types_breast,
            list_N=[2, 3, 5],
            proportions=[0.45, 0.45, 0.1]
        )
    )

# Generate
np.random.seed(42)
fov_dist_breast = FOVDistribution(
    frame_size=frame_size_breast,
    background_element=breast_stroma,
    other_elements=[breast_acinus, breast_vessel],
    elements_frequency=[0.75, 0.25],
    attempts_at_elements=[12, 4],
)

fov_breast = fov_dist_breast.generate_fov()

# Render
fig, ax = render_tissue(
    fov_breast, cell_props_breast,
    title='Mammary Gland: Acini and Stroma',
    legend_names=cell_type_names_breast,
    figsize=(14, 14),
    cell_alpha=0.75,
    background_color='#fff9f0',
)
plt.show()

---
## 4. Lymph Node with Follicles

Immune tissue with germinal centers and surrounding zones.

In [ ]:
# Lymph node parameters
n_cell_types_lymph = 7
frame_size_lymph = 1000

tissue_lymph = TissueCellTypes()
tissue_lymph.generate_types_and_markers(n_genes=55, n_cell_types=n_cell_types_lymph)
cell_type_names_lymph = ['Naive B', 'GC B cell', 'Plasma cell', 'T helper', 'T cytotoxic', 'DC', 'Macrophage']

# Immune cell colors
lymph_colors = {
    0: '#5DADE2',  # Naive B - Light blue
    1: '#2E86AB',  # GC B - Dark blue
    2: '#A23B72',  # Plasma - Magenta
    3: '#F18F01',  # T helper - Orange
    4: '#C73E1D',  # T cytotoxic - Red
    5: '#3A506B',  # DC - Dark slate
    6: '#6B2D5C',  # Macrophage - Purple
}

cell_props_lymph = CellTypesProperties(
    n_cell_types=n_cell_types_lymph,
    sizes=[8, 10, 14, 8, 10, 14, 16],
    anisotropy=[0.95, 0.9, 0.8, 0.95, 0.9, 0.7, 0.7],
    colordict=lymph_colors,
)

# Germinal center (follicle)
def germinal_center():
    rule = LayerRule(
        n_cell_types=n_cell_types_lymph,
        layer_types=[1, 0],  # GC B cells center, Naive B mantle
        layer_boundaries=[0.65],
        transition_width=15,
    )
    return ClusterElement(
        frame_size=frame_size_lymph,
        center=(np.random.uniform(150, 850), np.random.uniform(150, 850)),
        radius=80 + np.random.uniform(-20, 30),
        density_profile='gaussian',
        tipical_cell_spacing=8,
        rules=rule,
        n_cell_types=n_cell_types_lymph,
    )

# T cell zone background
def t_cell_zone():
    return FrameWideElement(
        frame_size=frame_size_lymph,
        tipical_cell_spacing=12,
        rules=MixOfNCellTypesRule(
            n_cell_types=n_cell_types_lymph,
            list_N=[3, 4, 5, 6],
            proportions=[0.35, 0.35, 0.15, 0.15]
        )
    )

# Generate
np.random.seed(42)
fov_dist_lymph = FOVDistribution(
    frame_size=frame_size_lymph,
    background_element=t_cell_zone,
    other_elements=[germinal_center],
    elements_frequency=[1.0],
    attempts_at_elements=6,
)

fov_lymph = fov_dist_lymph.generate_fov()

# Render
fig, ax = render_tissue(
    fov_lymph, cell_props_lymph,
    title='Lymph Node: Germinal Centers and T Cell Zone',
    legend_names=cell_type_names_lymph,
    figsize=(14, 14),
    cell_alpha=0.8,
    background_color='#f0f5ff',
)
plt.show()

---
## 5. Muscle with Vasculature

Skeletal muscle fibers with interspersed blood vessels.

In [ ]:
# Muscle parameters
n_cell_types_muscle = 5
frame_size_muscle = 1000

tissue_muscle = TissueCellTypes()
tissue_muscle.generate_types_and_markers(n_genes=40, n_cell_types=n_cell_types_muscle)
cell_type_names_muscle = ['Myocyte', 'Satellite Cell', 'Endothelial', 'Fibroblast', 'Macrophage']

# Muscle colors
muscle_colors = {
    0: '#C0392B',  # Myocyte - Deep red
    1: '#E74C3C',  # Satellite - Lighter red
    2: '#3498DB',  # Endothelial - Blue
    3: '#F7DC6F',  # Fibroblast - Yellow
    4: '#9B59B6',  # Macrophage - Purple
}

cell_props_muscle = CellTypesProperties(
    n_cell_types=n_cell_types_muscle,
    sizes=[20, 8, 10, 12, 12],
    anisotropy=[0.4, 0.9, 0.85, 0.75, 0.85],
    colordict=muscle_colors,
)

# Muscle fiber bundle
def muscle_fiber():
    return FibrillarStructure(
        frame_size=frame_size_muscle,
        center=(np.random.uniform(200, 800), np.random.uniform(200, 800)),
        orientation=np.random.uniform(-15, 15),
        n_fibers=8,
        fiber_length=600,
        fiber_width=35,
        fiber_spacing=15,
        waviness=0.05,
        tipical_cell_spacing=12,
        rules=MixOfNCellTypesRule(
            n_cell_types=n_cell_types_muscle,
            list_N=[0, 1],
            proportions=[0.85, 0.15]
        ),
        n_cell_types=n_cell_types_muscle,
    )

# Blood vessel
def muscle_vessel():
    return LinearLumenStructure(
        frame_size=frame_size_muscle,
        start_point=(np.random.uniform(100, 900), np.random.uniform(100, 900)),
        end_point=(np.random.uniform(100, 900), np.random.uniform(100, 900)),
        width=20,
        lumen_fraction=0.5,
        tipical_cell_spacing=8,
        rules=SingleTypeRule(n_cell_types=n_cell_types_muscle, cell_type_ix=2),
    )

# Connective tissue background
def muscle_stroma():
    return FrameWideElement(
        frame_size=frame_size_muscle,
        tipical_cell_spacing=40,
        rules=MixOfNCellTypesRule(
            n_cell_types=n_cell_types_muscle,
            list_N=[3, 4],
            proportions=[0.7, 0.3]
        )
    )

# Generate
np.random.seed(42)
fov_dist_muscle = FOVDistribution(
    frame_size=frame_size_muscle,
    background_element=muscle_stroma,
    other_elements=[muscle_fiber, muscle_vessel],
    elements_frequency=[0.75, 0.25],
    attempts_at_elements=[2, 5],
)

fov_muscle = fov_dist_muscle.generate_fov()

# Render
fig, ax = render_tissue(
    fov_muscle, cell_props_muscle,
    title='Skeletal Muscle with Vasculature',
    legend_names=cell_type_names_muscle,
    figsize=(14, 14),
    cell_alpha=0.75,
    background_color='#fff5f5',
)
plt.show()

---
## 6. Tumor Microenvironment

Complex tumor tissue with immune infiltration and stromal reaction.

In [ ]:
# Tumor parameters
n_cell_types_tumor = 8
frame_size_tumor = 1000

tissue_tumor = TissueCellTypes()
tissue_tumor.generate_types_and_markers(n_genes=70, n_cell_types=n_cell_types_tumor)
cell_type_names_tumor = ['Tumor', 'CAF', 'Endothelial', 'CD8+ T', 'CD4+ T', 'Macrophage', 'NK', 'DC']

# TME colors
tumor_colors = {
    0: '#1C1C1C',  # Tumor - Black
    1: '#D4AC0D',  # CAF - Gold
    2: '#2980B9',  # Endothelial - Blue
    3: '#C0392B',  # CD8+ T - Red
    4: '#E67E22',  # CD4+ T - Orange
    5: '#7D3C98',  # Macrophage - Purple
    6: '#16A085',  # NK - Teal
    7: '#5D6D7E',  # DC - Gray
}

cell_props_tumor = CellTypesProperties(
    n_cell_types=n_cell_types_tumor,
    sizes=[14, 16, 10, 8, 8, 14, 10, 12],
    anisotropy=[0.75, 0.6, 0.9, 0.95, 0.95, 0.7, 0.9, 0.7],
    colordict=tumor_colors,
)

# Tumor nest
def tumor_nest():
    return ClusterElement(
        frame_size=frame_size_tumor,
        center=(np.random.uniform(200, 800), np.random.uniform(200, 800)),
        radius=100 + np.random.uniform(-30, 50),
        density_profile='uniform',
        tipical_cell_spacing=8,
        rules=MixOfNCellTypesRule(
            n_cell_types=n_cell_types_tumor,
            list_N=[0],
            proportions=[1.0]
        ),
        n_cell_types=n_cell_types_tumor,
    )

# Immune aggregate
def immune_aggregate():
    return ClusterElement(
        frame_size=frame_size_tumor,
        center=(np.random.uniform(100, 900), np.random.uniform(100, 900)),
        radius=50 + np.random.uniform(-15, 25),
        density_profile='gaussian',
        tipical_cell_spacing=7,
        rules=MixOfNCellTypesRule(
            n_cell_types=n_cell_types_tumor,
            list_N=[3, 4, 5, 6, 7],
            proportions=[0.35, 0.25, 0.2, 0.1, 0.1]
        ),
        n_cell_types=n_cell_types_tumor,
    )

# Tumor stroma (CAF-rich)
def tumor_stroma():
    return FrameWideElement(
        frame_size=frame_size_tumor,
        tipical_cell_spacing=20,
        rules=MixOfNCellTypesRule(
            n_cell_types=n_cell_types_tumor,
            list_N=[1, 2, 3, 4, 5],
            proportions=[0.4, 0.15, 0.15, 0.15, 0.15]
        )
    )

# Generate
np.random.seed(42)
fov_dist_tumor = FOVDistribution(
    frame_size=frame_size_tumor,
    background_element=tumor_stroma,
    other_elements=[tumor_nest, immune_aggregate],
    elements_frequency=[0.6, 0.4],
    attempts_at_elements=[4, 6],
)

fov_tumor = fov_dist_tumor.generate_fov()

# Render
fig, ax = render_tissue(
    fov_tumor, cell_props_tumor,
    title='Tumor Microenvironment',
    legend_names=cell_type_names_tumor,
    figsize=(14, 14),
    cell_alpha=0.8,
    background_color='#f8f8f8',
)
plt.show()

---
## 7. Combined Gallery View

In [ ]:
# Create a combined gallery
fig, axes = plt.subplots(2, 3, figsize=(18, 12))

tissues = [
    (fov_colon, cell_props_colon, 'Colon'),
    (fov_cortex, cell_props_cortex, 'Cortex'),
    (fov_breast, cell_props_breast, 'Mammary'),
    (fov_lymph, cell_props_lymph, 'Lymph Node'),
    (fov_muscle, cell_props_muscle, 'Muscle'),
    (fov_tumor, cell_props_tumor, 'Tumor'),
]

for ax, (fov, props, name) in zip(axes.flat, tissues):
    # Apply properties if needed
    if not hasattr(fov, 'cell_colors') or fov.cell_colors is None:
        props.apply(fov)
    
    # Draw cells
    ellipses = []
    colors = []
    for i in range(len(fov.cell_centroids)):
        ellipse = Ellipse(
            xy=(fov.cell_centroids[i, 0], fov.cell_centroids[i, 1]),
            width=2 * fov.cell_major_axis[i],
            height=2 * fov.cell_minor_axis[i],
            angle=np.degrees(fov.cell_rotation[i]),
        )
        ellipses.append(ellipse)
        colors.append(fov.cell_colors[i])
    
    collection = PatchCollection(ellipses, alpha=0.7)
    collection.set_facecolors(colors)
    collection.set_edgecolors('black')
    collection.set_linewidths(0.2)
    ax.add_collection(collection)
    
    frame_size = fov.frame_size if hasattr(fov, 'frame_size') else 1000
    ax.set_xlim(0, frame_size)
    ax.set_ylim(0, frame_size)
    ax.set_aspect('equal')
    ax.set_title(f'{name}\n({len(fov.cell_centroids)} cells)', fontsize=12, fontweight='bold')
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_facecolor('#fafafa')

plt.suptitle('Tissue Gallery: Diverse Tissue Simulations with PointillSim', 
             fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---
## Summary

This gallery demonstrated PointillSim's ability to create diverse, beautiful tissue simulations:

| Tissue | Key Features | Structures Used |
|--------|--------------|------------------|
| **Colon** | Crypt architecture, stem cell gradients | VacuolatedStructure, LayerRule |
| **Cortex** | Layered neurons, smooth gradients | ProbabilityNodeFieldRule |
| **Mammary** | Acini, bilayer epithelium, adipocytes | VacuolatedStructure, LinearLumenStructure |
| **Lymph Node** | Germinal centers, T cell zones | ClusterElement |
| **Muscle** | Parallel fibers, vasculature | FibrillarStructure, LinearLumenStructure |
| **Tumor** | Nests, immune infiltration, CAFs | ClusterElement, MixOfNCellTypesRule |

### Tips for Beautiful Visualizations

1. **Use custom color palettes** that reflect tissue biology
2. **Vary cell sizes** to represent different cell types
3. **Adjust anisotropy** for elongated cells (muscle, fibroblasts)
4. **Add nuclei** for more realistic appearance
5. **Use appropriate backgrounds** that complement the tissue colors